# 대규모 언어 모델로 텍스트 생성하기

<table align="left"><tr><td>
<a href="https://colab.research.google.com/github/rickiepark/hg-mldl2/blob/main/10-3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="코랩에서 실행하기"/></a>
</td></tr></table>

EXAONE 3.5 모델은 `transformers` 최신 버전과 호환되지 않습니다. 따라서 이 절의 코드를 실행하기 위해 `transformers` 5.1.0 버전을 설치합니다.

In [2]:
!pip install -U transformers==5.1.0

In [3]:
# 깃허브에서 위젯 상태 오류를 피하기 위해 진행 표시줄을 나타내지 않도록 설정합니다.
from transformers.utils import logging

logging.disable_progress_bar()

## EXAONE-3.5로 상품 질문에 대한 대답 생성하기

In [4]:
from transformers import pipeline

pipe = pipeline(task="text-generation",
                model="LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct",
                device=0, trust_remote_code=True)

A new version of the following files was downloaded from https://huggingface.co/LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct:
- configuration_exaone.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct:
- modeling_exaone.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


In [5]:
messages = [
    {"role": "system",
     "content": "너는 쇼핑몰 홈페이지에 올라온 질문에 대답하는 Q&A 챗봇이야. \
                 확정적인 답변을 하지 말고 제품 담당자가 정확한 답변을 하기 위해 \
                 시간이 필요하다는 간단하고 친절한 답변을 생성해줘."},
    {"role": "user", "content": "이 다이어리에 내년도 공휴일이 표시되어 있나요?"}
]

pipe(messages, max_new_tokens=200)

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': [{'role': 'system',
    'content': '너는 쇼핑몰 홈페이지에 올라온 질문에 대답하는 Q&A 챗봇이야.                  확정적인 답변을 하지 말고 제품 담당자가 정확한 답변을 하기 위해                  시간이 필요하다는 간단하고 친절한 답변을 생성해줘.'},
   {'role': 'user', 'content': '이 다이어리에 내년도 공휴일이 표시되어 있나요?'},
   {'role': 'assistant',
    'content': '네, 저희가 확인해 보니 다이어리의 공휴일 일정은 주로 해당 연도의 공휴일 정보에 맞춰져 있습니다. 정확한 내용은 해당 연도의 공휴일 정보에 따라 달라질 수 있으니, 가장 확실한 정보를 얻으시려면 직접 고객센터에 연락주시거나, 방문하시어 확인하시는 것이 좋을 것 같습니다. 시간을 내주셔서 감사합니다! 추가 질문이 있으시면 언제든지 말씀해 주세요.'}]}]

In [6]:
pipe(messages, max_new_tokens=200, return_full_text=False)

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': '안녕하세요! 다이어리에 내년도 공휴일 정보가 포함되어 있는지에 대해 자세히 알려드리려면 현재 해당 상품의 정확한 모델 번호나 추가 정보가 필요합니다. 제품 담당자께서는 이러한 세부 사항을 확인하시고 가장 정확한 답변을 드릴 수 있을 것 같아요. 혹시 모델 번호나 구매 날짜를 알려주시면 도움이 될 것 같습니다! 감사합니다. 😊'}]

In [7]:
output = pipe(messages, max_new_tokens=200, return_full_text=False,
              do_sample=True)
print(output[0]['generated_text'])

Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


안녕하세요! 다이어리의 내년 공휴일 정보를 확인해드리지 못해 죄송합니다. 현재로선 정확한 답변을 드리기 어렵습니다. 제품 담당자님께서 바로 확인하시고 가장 정확한 정보를 제공해 주실 거예요. 조금만 기다려 주시면 곧 답변 드리겠습니다! 감사합니다.


## 토큰 디코딩 전략

### 기본 샘플링

In [8]:
import numpy as np

logits = np.array([1, 2, 3, 4, 100])

In [9]:
from scipy.special import softmax

probas = softmax(logits)
print(probas)

[1.01122149e-43 2.74878501e-43 7.47197234e-43 2.03109266e-42
 1.00000000e+00]


In [10]:
np.random.multinomial(100, probas)

array([  0,   0,   0,   0, 100])

In [11]:
probas = softmax(logits/100)
np.random.multinomial(100, probas)

array([16, 13,  8, 14, 49])

In [12]:
output = pipe(messages, max_new_tokens=200, return_full_text=False,
              do_sample=True, temperature=10.0)
print(output[0]['generated_text'])

Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


확인차 한말씀 부탁드렷숹? 정확하기 어려울 경우 관리자 연락처대로 연락주시고나면 도와놓구요 기다혀서 필요항 도움 알려 드리니 기다림 좀 참고 부탁드립니다✨ 감사 합니다!^^🏂❟ 😊


In [13]:
output = pipe(messages, max_new_tokens=200, return_full_text=False,
              do_sample=True, temperature=0.001)
print(output[0]['generated_text'])

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


안녕하세요! 다이어리에 내년의 공휴일이 미리 표시되어 있는지에 대해 정확한 답변을 드리기 위해서는 제품 담당자에게 확인이 필요합니다. 현재로선 직접 확인이 어려우니, 저희가 안내드릴 수 있는 방법으로는 고객센터에 연락하시거나, 제품 페이지 내의 문의 게시판을 통해 질문해 보시는 것이 좋을 것 같습니다. 담당자분께서 빠르게 답변해 주실 거예요! 감사합니다.


### top-k 샘플링

In [14]:
output = pipe(messages, max_new_tokens=200, return_full_text=False,
              do_sample=True, top_k=10)
print(output[0]['generated_text'])

Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'top_k'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


네, 맞아요! 저희 다이어리에는 내년도 공휴일이 자세히 표시되어 있답니다. 하지만 정확한 날짜와 내용을 확인하시려면 저희 고객센터에 연락주시거나, 직접 제품을 확인해 보시는 게 가장 정확할 것 같아요. 혹시 시간이 좀 걸리실 것 같아요, 정확한 정보를 드리기 위해 잠시만 기다려 주세요! 곧 답변 드리겠습니다. 감사합니다.


In [15]:
output = pipe(messages, max_new_tokens=200, return_full_text=False,
              do_sample=True, top_k=10, temperature=10.0)
print(output[0]['generated_text'])

Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'top_k', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


감사하요~! 질문 주님 덕분을 잊질 마세요. 현재 제공받은 제품 상세 안내로는내년도와 각 해당 국가들 공휴일에 대해 자세한 캘린더나 일정표 내용에 대한 정보만 확인해 주실 수 없는 점죄송합니다. 하지만 당사에 직접 연락이나 저희 쇼핑몰 이벤트 알림 구독 서비스로 공휴일 정보가 포함되지된 추가 이벤트 안내 받으시기엔 좋으시리라라 예상됬으니까요! 어떻게든 더 자세하고 빠른 정보 드리지 못해 죄송하지만 이해하고 부탁드립니다~! 😔


### top-p 샘플링

In [16]:
output = pipe(messages, max_new_tokens=200, return_full_text=False,
              do_sample=True, top_p=0.9)
print(output[0]['generated_text'])

Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'top_p'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


안녕하세요! 다이어리에 내년도 공휴일 정보가 포함되어 있는지에 대해 궁금하시군요. 현재로선 정확한 답변을 드리기 어렵습니다. 저희 측에서는 각 다이어리의 제작 연도와 함께 포함된 공휴일 정보를 확인하고 있지만, 그 정보가 최신 업데이트 되었는지에 따라 다를 수 있어요. 가장 확실한 답변을 위해서는 제품 담당자님께 직접 문의하시는 것이 좋을 것 같습니다. 담당자님께서 최신 정보를 바탕으로 정확하게 알려드릴 수 있을 거예요! 감사합니다.


In [17]:
output = pipe(messages, max_new_tokens=200, return_full_text=False,
              do_sample=True, temperature=0.8, top_k=100, top_p=0.9)
print(output[0]['generated_text'])

Passing `generation_config` together with generation-related arguments=({'top_p', 'top_k', 'temperature', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


안녕하세요! 다이어리에 내년의 공휴일이 미리 표시되어 있는지에 대해 궁금하시군요! 저희는 해당 정보를 미리 업데이트하긴 하지만, 정확한 공휴일 일정은 매년 조금씩 변동될 수 있어요. 현재로선 저희가 직접 확인해드릴 수는 없지만, 제품 담당자분께 문의하시면 가장 최신 정보를 드릴 수 있을 거예요. 담당자분께 연락하셔서 자세히 알아보시는 게 좋을 것 같아요! 감사합니다.


## GPT-4o로 상품 질문에 대한 대답 생성하기

In [19]:
from openai import OpenAI

client = OpenAI(api_key="Your OpenAPI Key")

completion = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages
)

In [20]:
print(completion.choices[0].message.content)

안녕하세요! 해당 다이어리에 내년도 공휴일이 포함되어 있는지 확인해보겠습니다. 제품 담당자에게 확인 후 정확한 답변을 드리겠습니다. 조금만 기다려 주세요! 감사합니다.


In [21]:
completion = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    top_p=0.9
)
print(completion.choices[0].message.content)

안녕하세요! 해당 다이어리에 내년도 공휴일이 표시되어 있는지 확인해보겠습니다. 제품 담당자에게 문의 후 정확한 답변을 드리도록 하겠습니다. 조금만 기다려 주세요!


In [22]:
completion = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    temperature=1.8
)
print(completion.choices[0].message.content)

고객님, 해당 다이어리에 내년도 공휴일이 표시되어 있는지 확인을 위해 저희 상품 담당자에게 몇 군데 문의해 보겠습니다.  잠시만 기다려 주시면 정확한 정보로 도와드리겠습니다. 감사합니다!
